In [1]:
import pandas as pd
import re
import nltk
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

nltk.download('stopwords')
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [2]:
resume_df = pd.read_csv("Resume.csv")
job_df = pd.read_csv("job_descriptions.csv")

In [3]:
print(resume_df.columns)
print(job_df.columns)

resume_df.head()
job_df.head()

Index(['ID', 'Resume_str', 'Resume_html', 'Category'], dtype='object')
Index(['Job Id', 'Experience', 'Qualifications', 'Salary Range', 'location',
       'Country', 'latitude', 'longitude', 'Work Type', 'Company Size',
       'Job Posting Date', 'Preference', 'Contact Person', 'Contact',
       'Job Title', 'Role', 'Job Portal', 'Job Description', 'Benefits',
       'skills', 'Responsibilities', 'Company', 'Company Profile'],
      dtype='object')


,Job Id,Experience,Qualifications,Salary Range,location,Country,latitude,longitude,Work Type,Company Size,...,Contact,Job Title,Role,Job Portal,Job Description,Benefits,skills,Responsibilities,Company,Company Profile
0,1089843540111562,5 to 15 Years,M.Tech,$59K-$99K,Douglas,Isle of Man,54.2361,-4.5481,Intern,26801,...,001-381-930-7517x737,Digital Marketing Specialist,Social Media Manager,Snagajob,Social Media Managers oversee an organizations...,"{'Flexible Spending Accounts (FSAs), Relocatio...","Social media platforms (e.g., Facebook, Twitte...","Manage and grow social media accounts, create ...",Icahn Enterprises,"{""Sector"":""Diversified"",""Industry"":""Diversifie..."
1,398454096642776,2 to 12 Years,BCA,$56K-$116K,Ashgabat,Turkmenistan,38.9697,59.5563,Intern,100340,...,461-509-4216,Web Developer,Frontend Web Developer,Idealist,Frontend Web Developers design and implement u...,"{'Health Insurance, Retirement Plans, Paid Tim...","HTML, CSS, JavaScript Frontend frameworks (e.g...","Design and code user interfaces for websites, ...",PNC Financial Services Group,"{""Sector"":""Financial Services"",""Industry"":""Com..."
2,481640072963533,0 to 12 Years,PhD,$61K-$104K,Macao,"Macao SAR, China",22.1987,113.5439,Temporary,84525,...,9687619505,Operations Manager,Quality Control Manager,Jobs2Careers,Quality Control Managers establish and enforce...,"{'Legal Assistance, Bonuses and Incentive Prog...",Quality control processes and methodologies St...,Establish and enforce quality control standard...,United Services Automobile Assn.,"{""Sector"":""Insurance"",""Industry"":""Insurance: P..."
3,688192671473044,4 to 11 Years,PhD,$65K-$91K,Porto-Novo,Benin,9.3077,2.3158,Full-Time,129896,...,+1-820-643-5431x47576,Network Engineer,Wireless Network Engineer,FlexJobs,"Wireless Network Engineers design, implement, ...","{'Transportation Benefits, Professional Develo...",Wireless network design and architecture Wi-Fi...,"Design, configure, and optimize wireless netwo...",Hess,"{""Sector"":""Energy"",""Industry"":""Mining, Crude-O..."
4,117057806156508,1 to 12 Years,MBA,$64K-$87K,Santiago,Chile,-35.6751,-71.5429,Intern,53944,...,343.975.4702x9340,Event Manager,Conference Manager,Jobs2Careers,A Conference Manager coordinates and manages c...,"{'Flexible Spending Accounts (FSAs), Relocatio...",Event planning Conference logistics Budget man...,Specialize in conference and convention planni...,Cairn Energy,"{""Sector"":""Energy"",""Industry"":""Energy - Oil & ..."


In [4]:
def clean_text(text):
    if pd.isnull(text):
        return ""

    text = text.lower()
    text = re.sub(r'\W', ' ', text)
    text = re.sub(r'\s+', ' ', text)

    words = text.split()
    words = [w for w in words if w not in stop_words]

    return " ".join(words)

In [5]:
resume_df['clean_resume'] = resume_df['Resume_str'].apply(clean_text)

# adapte le nom de la colonne si besoin
job_df['clean_job'] = job_df['Job Description'].apply(clean_text)

In [6]:
skills_list = [
    "python", "java", "machine learning", "data analysis",
    "sql", "excel", "deep learning", "nlp", "tensorflow",
    "pandas", "communication", "project management"
]

def extract_skills(text):
    found = []
    for skill in skills_list:
        if skill in text:
            found.append(skill)
    return found

resume_df['skills'] = resume_df['clean_resume'].apply(extract_skills)

In [7]:
job_text = job_df['clean_job'].iloc[0]

In [8]:
def get_job_by_title(title):
    results = job_df[job_df['Job Title'].str.contains(title, case=False, na=False)]
    return results.iloc[0]['clean_job']

job_text = get_job_by_title("Data Scientist")

In [9]:
vectorizer = TfidfVectorizer()

all_texts = resume_df['clean_resume'].tolist() + [job_text]
tfidf_matrix = vectorizer.fit_transform(all_texts)

job_vector = tfidf_matrix[-1]
resume_vectors = tfidf_matrix[:-1]

scores = cosine_similarity(resume_vectors, job_vector)

resume_df['score'] = scores

In [10]:
job_skills = set(extract_skills(job_text))

def missing_skills(cv_skills):
    return list(job_skills - set(cv_skills))

resume_df['missing_skills'] = resume_df['skills'].apply(missing_skills)

In [11]:
ranked_df = resume_df.sort_values(by='score', ascending=False)

top_candidates = ranked_df[['ID', 'score', 'skills', 'missing_skills']].head(10)

top_candidates

,ID,score,skills,missing_skills
1762,12011623,0.148194,"[python, machine learning, data analysis, sql,...",[]
2291,12777487,0.140328,[],[machine learning]
2153,34953092,0.112128,"[python, machine learning, sql]",[]
194,18835363,0.110869,[project management],[machine learning]
349,90363254,0.104212,[communication],[machine learning]
374,36206485,0.101883,"[excel, communication]",[machine learning]
1709,25608963,0.101547,"[excel, communication]",[machine learning]
1218,21156767,0.100963,"[python, java, machine learning, data analysis...",[]
845,22093368,0.100145,[excel],[machine learning]
1764,28923650,0.097322,[machine learning],[]


In [12]:
for i, row in top_candidates.iterrows():
    print("Candidate ID:", row['ID'])
    print("Score:", round(row['score'], 3))
    print("Skills:", row['skills'])
    print("Missing Skills:", row['missing_skills'])
    print("-" * 50)

Candidate ID: 12011623
Score: 0.148
Skills: ['python', 'machine learning', 'data analysis', 'sql', 'excel', 'pandas']
Missing Skills: []
--------------------------------------------------
Candidate ID: 12777487
Score: 0.14
Skills: []
Missing Skills: ['machine learning']
--------------------------------------------------
Candidate ID: 34953092
Score: 0.112
Skills: ['python', 'machine learning', 'sql']
Missing Skills: []
--------------------------------------------------
Candidate ID: 18835363
Score: 0.111
Skills: ['project management']
Missing Skills: ['machine learning']
--------------------------------------------------
Candidate ID: 90363254
Score: 0.104
Skills: ['communication']
Missing Skills: ['machine learning']
--------------------------------------------------
Candidate ID: 36206485
Score: 0.102
Skills: ['excel', 'communication']
Missing Skills: ['machine learning']
--------------------------------------------------
Candidate ID: 25608963
Score: 0.102
Skills: ['excel', 'communi

essaie pdf

In [13]:
pip install PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 5.0 MB/s eta 0:00:00


## **PDF READING**

In [14]:
import PyPDF2

def extract_text_from_pdf(file_path):
    text = ""

    with open(file_path, "rb") as file:
        reader = PyPDF2.PdfReader(file)

        for page in reader.pages:
            text += page.extract_text()

    return text

**TEST WITH A PDF**

In [15]:
pdf_text = extract_text_from_pdf("/content/31926950.pdf")

print(pdf_text[:1000])  # afficher un extrait

STAFFING COORDINATOR
Professional Summary
An energetic staffing professional seeking challenging experiences in Talent Acquisition and Talent Management. Solid communication,
interpersonal, and organizational skills. Experience in working with upper management and executives to coordinate meetings, travel arrangements
and onboarding of new employees.
Skill Highlights
Event Coordination
Microsoft Word, Excel, Power
Point, Outlook, SharePoint
BrassRing and Taleo
Candidate tracking systems
Training and experience in Infovision II, Retail Link, and Spectra 
databases; used to analyze sales
numbers and performance, and create 
progressive goals for upcoming months.
Professional Experience
07/2015
 
to 
Current
Staffing Coordinator
 
Company Name
 
ï¼​ 
City
 
, 
State
Partner with US Staffing Representatives by scheduling interviews for candidates located in the US and abroad.
Coordinated travel arrangements for domestic candidates while maintaining HR data through Taleo (BaxTalent) Systems

**APPLY THE MODEL TO THE PDF**

In [16]:
cv_clean = clean_text(pdf_text)

vectorizer = TfidfVectorizer()
vectors = vectorizer.fit_transform([cv_clean, job_text])

score = cosine_similarity(vectors[0:1], vectors[1:2])[0][0]

print("Score du CV PDF:", score)

Score du CV PDF: 0.035396591643989894


**MULTIPLE PDF**

In [17]:
#from google.colab import files
#uploaded = files.upload()

In [18]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [19]:
import os

os.listdir('/content/drive/MyDrive')

['VOULE-FRITITI_Yedidia.zip',
 "Capture d'écran 2023-12-14 160512.png",
 "Capture d'écran 2023-12-14 162811.png",
 "Capture d'écran 2023-12-14 163311.png",
 'Colab Notebooks',
 'Document sans titre.gdoc',
 'Classroom',
 'Accra Newtown_pt',
 'Voici la methodologie du travail qui a ete fait:....gsheet',
 'data_memoire',
 'data']

In [20]:
folder_path = "/content/drive/MyDrive/data"

In [21]:
def skill_match_score(cv_skills, job_skills):
    if len(job_skills) == 0:
        return 0
    return len(set(cv_skills) & set(job_skills)) / len(job_skills)

In [23]:
results = []

for root, dirs, files in os.walk(folder_path):
    for file in files:
        if file.lower().endswith(".pdf"):
            path = os.path.join(root, file)

            text = extract_text_from_pdf(path)

            if not text:
                continue

            clean = clean_text(text)
            cv_skills = extract_skills(clean)

            vectorizer = TfidfVectorizer()
            vectors = vectorizer.fit_transform([clean, job_text])

            tfidf_score = cosine_similarity(vectors[0:1], vectors[1:2])[0][0]

            job_skills = set(extract_skills(job_text))

            skill_score = skill_match_score(cv_skills, job_skills)

            final_score = 0.7 * tfidf_score + 0.3 * skill_score

            missing = list(job_skills - set(cv_skills))

            results.append({
                "file": file,
                "score": final_score,
                "skills": cv_skills,
                "missing_skills": missing
            })

results = sorted(results, key=lambda x: x['score'], reverse=True)

for r in results[:20]:
    print("📄 CV:", r["file"])
    print("⭐ Score:", round(r["score"], 2))
    print("✅ Skills:", r["skills"])
    print("❌ Missing Skills:", r["missing_skills"])
    print("-" * 30)

📄 CV: 21156767.pdf
⭐ Score: 0.41
✅ Skills: ['python', 'java', 'machine learning', 'data analysis', 'sql']
❌ Missing Skills: []
------------------------------
📄 CV: 12011623.pdf
⭐ Score: 0.41
✅ Skills: ['python', 'machine learning', 'data analysis', 'sql', 'excel', 'pandas']
❌ Missing Skills: []
------------------------------
📄 CV: 28505854.pdf
⭐ Score: 0.36
✅ Skills: ['java', 'machine learning', 'tensorflow']
❌ Missing Skills: []
------------------------------
📄 CV: 34953092.pdf
⭐ Score: 0.36
✅ Skills: ['python', 'machine learning', 'sql']
❌ Missing Skills: []
------------------------------
📄 CV: 22946204.pdf
⭐ Score: 0.36
✅ Skills: ['python', 'java', 'machine learning', 'data analysis', 'sql', 'excel']
❌ Missing Skills: []
------------------------------
📄 CV: 28923650.pdf
⭐ Score: 0.35
✅ Skills: ['machine learning']
❌ Missing Skills: []
------------------------------
📄 CV: 12351749.pdf
⭐ Score: 0.35
✅ Skills: ['machine learning', 'data analysis', 'excel', 'communication']
❌ Missing Sk